In [ ]:
%pip install scipy
# %pip install dataclasses
%pip install pandas
%pip install numpy
%pip install torch
%pip install wandb
%pip install datasets
%pip install transformers
%pip install huggingface_hub
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00


In [ ]:
# %%
import os
from datetime import datetime
current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Current time: {current_time_str}")
# %%
from typing import Any, Dict, List

import numpy as np
import torch
import wandb

from datasets import (
    load_dataset,
    Audio,
    load_from_disk
)
from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    AutoConfig,
    TrainingArguments,
    Trainer
)

from huggingface_hub import login
import evaluate
from google.colab import userdata

print("Check if GPU available:")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.get_device_name(): {torch.cuda.get_device_name()}")


# %%
# login to Hugging Face
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_TOKEN = userdata.get('WANDB_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable not set")
login(token=HF_TOKEN)

print("Logged in to Hugging Face successfully!")

# %%
# login to WANDB

if not WANDB_TOKEN:
    raise ValueError("WANDB_TOKEN environment variable not set")
wandb.login(key=WANDB_TOKEN)

# %%
# model_id = "facebook/mms-300m"
model_id = "utter-project/mHuBERT-147"
# model_id = "facebook/wav2vec2-xls-r-300m"

# %%
batch_size = 16
gradient_accumulation_steps = 8
num_train_epochs = 10
lr = 0.00002646
unfreeze_at_step = 500
max_duration = 5 # in seconds
label_smoothing = 0.2
weight_decay = 0.05
sampling_rate = 16000
warmup_ratio = 0.15
backbone_lr = 1e-5
head_lr = 0.003
early_stopping_patience = 2
training_schedule = 'cosine'

do_apply_dropout = True
hidden_dropout = 0.01           # Dropout for hidden states
attention_dropout = 0.01        # Dropout in attention layers
activation_dropout = 0.01       # Dropout after activation functions
feat_proj_dropout = 0.01

# %%

AVAILABLE_CPU = userdata.get("REQUEST_CPU")
if not AVAILABLE_CPU:
    import multiprocessing
    AVAILABLE_CPU = multiprocessing.cpu_count()
else:
  AVAILABLE_CPU = int(AVAILABLE_CPU)


# %%
WANDB_TEAM = userdata.get("WANDB_TEAM")
if not WANDB_TEAM:
    raise ValueError("WANDB_TEAM environment variable not set")

WANDB_PROJECT = userdata.get("WANDB_PROJECT")
if not WANDB_PROJECT:
    raise ValueError("WANDB_PROJECT environment variable not set")


wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_TEAM,
    name=f"SLID_{model_id}_{lr}_{current_time_str}",
    config={
        "model_id": model_id,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "num_train_epochs": num_train_epochs,
        "learning_rate": lr,
        "backbone_lr": backbone_lr,          # Monitor differential LR
        "head_lr": head_lr,              # Monitor differential LR
        "max_duration": max_duration, # Critical: how much context does the model get?
        "unfreeze_at_step": unfreeze_at_step,      # Note the inflection point
        "label_smoothing": label_smoothing,
        "optimizer": "AdamW",
        "sampling_rate": sampling_rate,
        "weight_decay": weight_decay,
        "warmup_ratio": warmup_ratio,
    }
)


# %%
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id,
    do_normalize=True,
    return_attention_mask=True,
)

# %%
dataset = load_dataset("badrex/nnti-dataset-full")

# %%
# check the strucutre of the dataset object
print(f"dataset['train']: {dataset['train']}")

# %%
# check the strucutre of one training sample (before decoding)
print(f"dataset['train'][0]: {dataset['train'][0]}")

# %%
# shuffle the dataset
train_ds = dataset['train'].shuffle(seed=42)
valid_ds = dataset['validation'].shuffle(seed=42)

# resample to 16kHz
train_ds = train_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))


# %%
# based on the model typel, set input features key
if model_id == "facebook/w2v-bert-2.0":
    input_features_key = "input_features"
else:
    input_features_key = "input_values"

# %%
# get the set of languages
LABELS = train_ds.unique('language')

sorted_labels = sorted(l.upper() for l in LABELS)
print(f"Languages: {sorted_labels}")

str_to_int = {
    s: i for i, s in enumerate(LABELS)
}


# %%
def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio_filepath"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        truncation=True,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        return_attention_mask=True,
    )

    inputs["label"] = [str_to_int[x] for x in examples["language"]]

    # convert input_features_key contains numerical arrays
    inputs[input_features_key] = [
        np.array(x) for x in inputs[input_features_key]
    ]

    inputs["length"] = [len(f) for f in inputs[input_features_key]]

    return inputs

# %%
keep_cols = ['speaker_id', 'language']

#
# ## encode the train and valid splits
# Using cached encoded data, in case if we change the preprocessing step
# - we will map it again
cache_name = f"encoded_ds_{model_id.split('/')[-1]}_{max_duration}sec"
save_path = os.path.join("./processed_data", cache_name)
if os.path.exists(save_path):
    print(f"Loading pre-processed data from {save_path}...")
    train_ds_encoded = load_from_disk(save_path + '/train_encoded')
    valid_ds_encoded = load_from_disk(save_path + '/valid_encoded')
else:
    print("Pre-processing data for the first time...")
    train_ds_encoded = train_ds.map(
        preprocess_function,
        batched=True,
        batch_size=32,
        num_proc=AVAILABLE_CPU,
        remove_columns=[c for c in train_ds.column_names if c not in keep_cols],
    )
    train_ds_encoded.save_to_disk(save_path + '/train_encoded')
    valid_ds_encoded = valid_ds.map(
        preprocess_function,
        remove_columns=[c for c in valid_ds.column_names if c not in keep_cols],
        batched=True,
        batch_size=32,
        num_proc=AVAILABLE_CPU,
    )
    valid_ds_encoded.save_to_disk(save_path + '/valid_encoded')


# %%
int_to_str = {
    i: s for s, i in str_to_int.items()
}

num_labels = len(int_to_str)

# %%
config = AutoConfig.from_pretrained(model_id)

config.num_labels=num_labels
config.label2id=str_to_int
config.id2label=int_to_str



# check if dropout is enabled
if do_apply_dropout:
    config.hidden_dropout = hidden_dropout
    config.attention_dropout = attention_dropout
    config.activation_dropout = activation_dropout
    config.feat_proj_dropout = feat_proj_dropout

# %%
# spoken language ID (SLID) model
slid_model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    config=config,
)

# %%
# create collator for padding
class AudioDataCollator:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # prepare the batch dict in the format expected by the feature extractor
        batch = {
            input_features_key: [f[input_features_key] for f in features],
            "attention_mask": [f["attention_mask"] for f in features]
        }

        # use the feature extractor's native padding
        batch = self.feature_extractor.pad(
            batch,
            padding=True,
            return_tensors="pt"
        )

        # add labels
        batch["labels"] = torch.tensor(
            [f["label"] for f in features],
            dtype=torch.long
        )

        return batch

from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# %%
# Differential Learning Rates
def get_optimizer_and_scheduler(model, training_args, num_training_steps):
    backbone_params = [p for n, p in model.named_parameters()
                       if "classifier" not in n and "projector" not in n]
    head_params = [p for n, p in model.named_parameters()
                   if "classifier" in n or "projector" in n]

    optimizer = AdamW([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params,     "lr": head_lr},
    ], weight_decay=weight_decay)

    num_warmup_steps = int(0.1 * num_training_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    return optimizer, scheduler

# %%
data_collator = AudioDataCollator(feature_extractor)

# %%
training_args = TrainingArguments(
    group_by_length=False,
    dataloader_num_workers=AVAILABLE_CPU,
    dataloader_pin_memory=True,
    #run_name='SLID_1',
    report_to="wandb",  # enable logging to W&B
    logging_steps=1,  # how often to log to W&B
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    learning_rate=lr,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    warmup_steps=warmup_ratio,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,  # True if your metric should be maximized (like accuracy)
    save_total_limit=2,  # Keep only the best model
    fp16=True,
    push_to_hub=False,
    label_smoothing_factor=label_smoothing # Since the acoustic problem is genuinely hard
)

# %%
# load evaluation metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(
        predictions=predictions,
        references=eval_pred.label_ids
    )

from transformers import TrainerCallback

# Freeze the backbone initially
for param in slid_model.hubert.parameters():  # or .wav2vec2 depending on model
    param.requires_grad = False

class UnfreezeCallback(TrainerCallback):
    def __init__(self, unfreeze_at_step):
        self.unfreeze_at_step = unfreeze_at_step

    def on_step_begin(self, args, state, control, model=None, **kwargs):
        if state.global_step == self.unfreeze_at_step:
            for param in model.parameters():
                param.requires_grad = True
            print(f"Backbone unfrozen at step {state.global_step}")

from transformers import EarlyStoppingCallback

# Calculate steps
steps_per_epoch = len(train_ds_encoded) // (batch_size * gradient_accumulation_steps)
num_training_steps = steps_per_epoch * num_train_epochs
optimizer, scheduler = get_optimizer_and_scheduler(
    slid_model, training_args, num_training_steps
)

# %%
trainer = Trainer(
    slid_model,
    training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=valid_ds_encoded,
    processing_class=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
    callbacks=[
        UnfreezeCallback(unfreeze_at_step=unfreeze_at_step)
    ]
)

# %%
print(f"Model is on device: {slid_model.device}")
wandb.watch(slid_model, log="gradients", log_freq=100)
print("Train loop starting...")
trainer.train()

# %%
# push model to hub
slid_model.push_to_hub(
    "radioduran/indic-language-identification"
)


# %%
print("Final evaluation starting...")
trainer.evaluate()

predictions = trainer.predict(valid_ds_encoded)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    probs=None,
    y_true=labels,
    preds=preds,
    class_names=sorted_labels
)})


# save model to disk
save_dir = "./indic-SLID/inprogress"
slid_model.save_pretrained(save_dir)


Current time: 20260226_151626
Check if GPU available:
torch.cuda.is_available(): True
torch.cuda.get_device_name(): Tesla T4
Logged in to Hugging Face successfully!


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


data/train-00000-of-00002.parquet:   0%|          | 0.00/382M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/311M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8689 [00:00<?, ? examples/s]

... (18 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final evaluation starting...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print("Завершаю работу рантайма...")
from google.colab import runtime
runtime.unassign()

Завершаю работу рантайма...
